# Grading Agent — the pipeline, box by box

One file. Each section below is one box in the flow diagram, in order, and each
one defines the piece of code that box represents. Nothing is imported from
elsewhere in the repo — everything this notebook needs is in this notebook.

```
BOX 1  Teacher/Website has module      →  §1
BOX 2  Student submits                 →  §2
BOX 3  Offline pre-processing          →  §3   (no model, no raw media leaves here)
       ├── video     → delivery signals    §3a
       ├── handwriting → OCR + completeness %  §3b
       └── reading   → speech-to-text       §3c
BOX 4  Grading Agent                   →  §4   (OpenAI Agents SDK)
BOX 5  Score + Evidence + Feedback     →  §5
BOX 6  Stored in DB                    →  §6
BOX 7  Show on Dashboard               →  §7
       Run it end to end               →  §8
```

**How to work in here.** Run §0 once, then the sections in order. Each box is
self-contained: to change how handwriting is read, you edit §3b and nothing
else. Re-run a section's cell and the later sections pick up the change.

§1 to §3 and §5 to §7 run with no API key. §4 and §8 call a model.

## §0 — Setup

Same harness as the team's existing agent notebook: `openai-agents` for the SDK,
`python-dotenv` for the key.

Make a file called `.env` next to this notebook with one line:

```
OPENAI_API_KEY=sk-...
```

Then add `.env` to `.gitignore`. The key never appears in the notebook, in its
saved output, or in a commit.

In [2]:
#%pip install openai-agents
#%pip install python-dotenv

In [3]:
from __future__ import annotations

import json, os, re, shutil, subprocess
from dataclasses import dataclass, field, asdict
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Literal, Optional

WORKDIR = Path.cwd()
DATA    = WORKDIR / "pipeline_data"          # everything this notebook writes
DATA.mkdir(exist_ok=True)

def has(cmd: str) -> bool:
    """Is this command-line tool available on this machine?"""
    return shutil.which(cmd) is not None

from dotenv import load_dotenv
load_dotenv(WORKDIR / ".env")          # explicit path: works the same in Jupyter,
                                       # in VS Code, and when run as a script

print("Working in:", WORKDIR)
print("ffmpeg installed:", has("ffmpeg"), "| ffprobe installed:", has("ffprobe"))
print("OPENAI_API_KEY loaded:", bool(os.environ.get("OPENAI_API_KEY")))

ModuleNotFoundError: No module named 'dotenv'

## §1 — BOX 1 · Teacher/Website has module

> *lesson + homework + grading instructions*

The module is the answer key. Everything downstream is measured against it, so
this is the one piece that must never be guessed at — if we do not have the real
module, the honest output is "cannot evaluate", not an invented score.

`grading_instructions` is deliberately free text and deliberately **per module**.
The agent reads it fresh on every submission rather than us hardcoding a rubric,
because a module asking for a calm encouraging voice is graded differently from a
plain read-aloud.

In [1]:
@dataclass
class Module:
    module_id: str
    title: str
    grading_instructions: str          # the rubric for THIS module, read live
    article: str = ""                  # what the student reads aloud
    word_list: list[tuple[str, str]] = field(default_factory=list)   # (english, hindi)
    expects_note: bool = True

    @property
    def answer_key(self) -> dict[str, str]:
        return {en.strip().lower(): hi for en, hi in self.word_list}


# ---- BOX 1 input -----------------------------------------------------------
# Replace this with a real fetch from the portal when we have it:
#   def load_module(module_id) -> Module:  ...read from the module page...

MODULE = Module(
    module_id="day12_task2",
    title="Day 12 – Task 2",
    grading_instructions=(
        "Read the article aloud at a comfortable, steady pace. "
        "Then write each English word with its Hindi meaning. "
        "Mark pace on whether a listener could follow, not on speed alone."
    ),
    article=("Delegating tasks helps a leader share responsibility with the team "
             "and build trust over time."),
    word_list=[
        ("delegate",       "सौंपना"),
        ("responsibility", "जिम्मेदारी"),
        ("trust",          "भरोसा"),
    ],
)

print(MODULE.title)
print("Instructions:", MODULE.grading_instructions)
print("Answer key:  ", MODULE.answer_key)

NameError: name 'dataclass' is not defined

## §2 — BOX 2 · Student submits

> *video / handwritten note / text*

`student_id` comes from the portal session the files were uploaded under. We
never work out who a child is from the content of their video — that is both
unreliable and not something we should be doing.

Any of the three can be absent. That is normal, and §4 grades what is there
rather than blocking on what is missing.

In [ ]:
@dataclass
class Submission:
    submission_id: str
    student_id: str
    module_id: str
    video_path: Optional[str] = None       # a recording of them reading aloud
    note_image_path: Optional[str] = None  # a photo of the handwritten page
    text: Optional[str] = None             # typed answer, if the portal allows it
    submitted_at: str = ""

    def __post_init__(self):
        if not (self.video_path or self.note_image_path or self.text):
            raise ValueError(
                f"{self.submission_id}: nothing was submitted. An empty submission "
                "is a portal bug, not a homework to grade.")


# ---- BOX 2 input -----------------------------------------------------------
SUBMISSION = Submission(
    submission_id="sub-001",
    student_id="stu-114",
    module_id="day12_task2",
    video_path=None,           # put a real path here to run speech-to-text (§3c)
    note_image_path=None,      # put a real path here to run OCR (§3b)
    text=("Delegating tasks helps a leader share responsibility with the team "
          "and build trust over time."),   # a typed answer, so this runs with no files
    submitted_at=datetime.now(timezone.utc).isoformat(timespec="seconds"),
)

print(SUBMISSION)

## §3 — BOX 3 · Offline pre-processing

> *no LLM, no raw video to agent*

This is the box that makes the whole design safe. Everything here runs on our own
machine, and what comes out the other side is **derived values only** — a
transcript, some counts, a percentage. The frames and the audio never leave this
section.

That is a hard line from the v1 spec for minors' data, and §4 is built so that it
holds structurally: the agent is handed the `PreProcessed` object below and has
no route to a file path.

In [ ]:
@dataclass
class PreProcessed:
    """Everything the agent is allowed to see. Derived values, never media."""
    # from the video
    transcript: Optional[str] = None
    duration_seconds: Optional[float] = None
    word_count: Optional[int] = None
    words_per_minute: Optional[float] = None
    has_audio: Optional[bool] = None
    delivery_signals: dict[str, float] = field(default_factory=dict)

    # from the handwritten page
    note_rows: dict[str, str] = field(default_factory=dict)      # english -> hindi written
    row_confidence: dict[str, float] = field(default_factory=dict)
    rows_expected: int = 0
    rows_filled: int = 0
    legible: Optional[bool] = None

    # from typed text
    typed_text: Optional[str] = None

    # what we could NOT do, and why. This matters as much as what we could.
    gaps: list[str] = field(default_factory=list)

    @property
    def completeness_pct(self) -> Optional[int]:
        if not self.rows_expected:
            return None
        return round(100 * self.rows_filled / self.rows_expected)

    @property
    def unclear_rows(self) -> list[str]:
        return sorted(w for w, c in self.row_confidence.items() if c < 0.75)

### §3a — video → delivery signals

**Deprioritised in v1.** The spec puts gesture, eye contact and pronunciation out
of scope for now, so this returns nothing and says so.

It is left here as a named seam rather than deleted, because the shape of what it
would return is the important part: a dict of *numbers*. If this is ever built,
frames are sampled and scored here, on our machine, and only the resulting
numbers move forward. A frame must never become a value in `PreProcessed`.

In [ ]:
def extract_delivery_signals(video_path: str) -> tuple[dict[str, float], list[str]]:
    """Derived delivery numbers from the video. Nothing here returns an image.

    Returns (signals, gaps).
    """
    # TODO(v2): sample frames locally, score them locally, return numbers only.
    #   signals = {"hands_visible_share": 0.62, "facing_camera_share": 0.81}
    return {}, ["Delivery signals from video are out of scope for v1."]

### §3b — handwriting → OCR + completeness %

Two numbers that look similar and must stay separate:

- **completeness** is how many rows were filled in, counted *without reading
  them*. That keeps it honest even when the OCR was poor.
- **confidence** is how sure we are we read a given row correctly. A row below
  the threshold **abstains** rather than being marked wrong — a student who
  corrected their own mistake must not be the one who pays for the smudge.

In [ ]:
def read_handwriting(image_path: str, module: Module) -> tuple[dict, list[str]]:
    """Read the handwritten page. Returns (fields, gaps).

    Swap the body for a real OCR call — Tesseract, a vision API, whatever the
    team picks. The contract is what matters: rows read, a confidence per row,
    a count of rows filled, and an honest `legible` flag.
    """
    gaps: list[str] = []

    # TODO(you): real OCR goes here.
    #   from PIL import Image; import pytesseract
    #   raw = pytesseract.image_to_data(Image.open(image_path), lang="hin+eng", ...)
    gaps.append("OCR is not wired up yet — no handwriting was read.")

    return {
        "note_rows": {},
        "row_confidence": {},
        "rows_expected": len(module.word_list),
        "rows_filled": 0,
        "legible": None,           # None = we do not know, which is not False
    }, gaps

### §3c — reading → speech-to-text

Two things to notice.

The audio is **extracted before uploading**. A three-minute phone video is around
26 MB, well past Whisper's 25 MB limit, but the speech inside it is under 2 MB.
Sending the video whole simply fails.

And three outcomes are kept distinct, because confusing them mislabels a child:
*no video submitted*, *a video with no sound* (the microphone was off — the
student can fix that), and *a video we failed to transcribe* (our problem, and no
reason to tell anyone they recorded in silence).

In [ ]:
def probe_video(video_path: str) -> tuple[Optional[float], bool]:
    """(duration_seconds, has_audio_stream) via ffprobe."""
    if not has("ffprobe"):
        return None, False
    def run(args):
        out = subprocess.run(["ffprobe", "-v", "error", *args, "-of", "json", video_path],
                             capture_output=True, text=True)
        return json.loads(out.stdout or "{}")
    dur = run(["-show_entries", "format=duration"]).get("format", {}).get("duration")
    aud = run(["-select_streams", "a", "-show_entries", "stream=codec_type"]).get("streams", [])
    return (float(dur) if dur else None), bool(aud)


def extract_audio(video_path: str) -> str:
    """16 kHz mono audio beside the video. Keeps the upload under the limit."""
    out = str(DATA / (Path(video_path).stem + ".m4a"))
    subprocess.run(["ffmpeg", "-y", "-i", video_path, "-vn",
                    "-ac", "1", "-ar", "16000", "-b:a", "64k", out],
                   capture_output=True, check=True)
    return out


def speech_to_text(video_path: str) -> tuple[dict, list[str]]:
    """Transcribe the reading. Returns (fields, gaps)."""
    gaps: list[str] = []
    duration, audio_present = probe_video(video_path)

    if not has("ffmpeg"):
        return {"duration_seconds": duration, "has_audio": audio_present}, \
               ["ffmpeg is not installed, so no audio could be extracted."]

    if not audio_present:
        # The student's to fix, and they should be told kindly.
        return {"duration_seconds": duration, "has_audio": False}, \
               ["The recording carries no audio track."]

    try:
        from openai import OpenAI
        with open(extract_audio(video_path), "rb") as fh:
            text = OpenAI().audio.transcriptions.create(model="whisper-1", file=fh).text
    except Exception as exc:
        # OUR gap. Never reported to the student as silence.
        return {"duration_seconds": duration, "has_audio": True}, \
               [f"Transcription failed ({type(exc).__name__}). This is our gap, "
                f"not the student's."]

    words = len(text.split())
    wpm = round(words / (duration / 60), 1) if duration else None
    return {"transcript": text, "duration_seconds": duration, "has_audio": True,
            "word_count": words, "words_per_minute": wpm}, gaps

### §3 — putting box 3 together

One function, called once per submission. Anything a branch could not do lands in
`gaps`, and §4 is told about it explicitly.

In [ ]:
def preprocess(sub: Submission, module: Module) -> PreProcessed:
    pre = PreProcessed(typed_text=sub.text)

    if sub.video_path:
        fields, gaps = speech_to_text(sub.video_path)          # §3c
        for k, v in fields.items():
            setattr(pre, k, v)
        pre.gaps += gaps

        signals, gaps = extract_delivery_signals(sub.video_path)  # §3a
        pre.delivery_signals = signals
        pre.gaps += gaps
    else:
        pre.gaps.append("No video was submitted.")

    if sub.note_image_path:
        fields, gaps = read_handwriting(sub.note_image_path, module)  # §3b
        for k, v in fields.items():
            setattr(pre, k, v)
        pre.gaps += gaps
    else:
        pre.rows_expected = len(module.word_list)
        pre.gaps.append("No handwritten page was submitted.")

    return pre


PRE = preprocess(SUBMISSION, MODULE)

print("transcript:      ", (PRE.transcript or "—")[:70])
print("words per minute:", PRE.words_per_minute)
print("completeness:    ", f"{PRE.completeness_pct}%" if PRE.completeness_pct is not None else "—")
print("unclear rows:    ", PRE.unclear_rows or "—")
print("\ngaps:")
for g in PRE.gaps:
    print("  -", g)

> With no video and no photo attached, every branch reports a gap. That is the
> correct output, not a failure — and §4 will grade what is there and say what is
> missing rather than inventing numbers.
>
> To see it work end to end, put real paths into `SUBMISSION` in §2 and re-run
> §2 and §3.

## §4 — BOX 4 · The Grading Agent

Built on the **OpenAI Agents SDK**. Four pieces, in this order:

1. **The output shape** — the model cannot reply with prose. It fills in
   `GradingResult`, and every score carries the evidence behind it.
2. **The tools** — what it may go and find out. Note that none of them takes a
   file path: the agent is closed over the `PreProcessed` object from §3, so the
   privacy line from box 3 holds however the prompt is edited later.
3. **The instructions**.
4. **The runner**.

### §4.1 — what the model must return

In [ ]:
from pydantic import BaseModel, Field

class Score(BaseModel):
    parameter: str
    score: Optional[int] = Field(None, description="1-5, or null if you could not judge it")
    evidence: str = Field(description="What you observed, and which part of the "
                                      "module's instructions it is measured against")
    abstained_because: Optional[str] = None


class GradingResult(BaseModel):
    scores: list[Score]
    completeness_note: str = Field(
        description="How much of the assignment was attempted, in plain words. "
                    "Reported, never used to lower a score.")
    feedback_en: str = Field(description="2-4 short sentences. Warm. Strength first.")
    feedback_hi: str = Field(description="The same message in Hindi")
    for_the_teacher: str = Field("", description="Not shown to the student")


# What v1 grades. Edit this list to change scope.
PARAMETERS = ("speed", "vocabulary", "completion", "accuracy")

### §4.2 — the tools

Each one returns a JSON string. `build_tools` closes over `module` and `pre`, so
the agent reads the values §1 and §3 produced and cannot reach anything else.

In [ ]:
def build_tools(module: Module, pre: PreProcessed):
    from agents import function_tool

    @function_tool
    def get_module() -> str:
        """The module's grading instructions, article and word list. Read this first."""
        return json.dumps({
            "title": module.title,
            "grading_instructions": module.grading_instructions,
            "article": module.article,
            "word_list": [{"english": e, "expected_hindi": h} for e, h in module.word_list],
        }, ensure_ascii=False)

    @function_tool
    def get_reading() -> str:
        """What the student said, with its timing. Says plainly when there is nothing."""
        return json.dumps({
            "transcript": pre.transcript,
            "duration_seconds": pre.duration_seconds,
            "word_count": pre.word_count,
            "words_per_minute": pre.words_per_minute,
            "has_audio": pre.has_audio,
        }, ensure_ascii=False)

    @function_tool
    def get_handwriting() -> str:
        """The page as read, and how much of it was filled in."""
        return json.dumps({
            "rows_expected": pre.rows_expected,
            "rows_filled": pre.rows_filled,
            "percent_attempted": pre.completeness_pct,
            "rows": pre.note_rows,
            "unclear_rows": pre.unclear_rows,
            "legible": pre.legible,
        }, ensure_ascii=False)

    @function_tool
    def get_typed_answer() -> str:
        """Anything the student typed, if the portal allowed a text answer."""
        return json.dumps({"typed_text": pre.typed_text}, ensure_ascii=False)

    @function_tool
    def get_gaps() -> str:
        """What pre-processing could NOT do, and why. Read this before scoring.

        A gap on our side is never the student's fault and must never lower a score.
        """
        return json.dumps({"gaps": pre.gaps}, ensure_ascii=False)

    @function_tool
    def check_hindi(english_word: str, written: str) -> str:
        """Compare one written Hindi answer against the module's key."""
        expected = module.answer_key.get(english_word.strip().lower())
        if expected is None:
            return json.dumps({"error": "That word is not in this module."})
        norm = lambda s: re.sub(r"[\u093c\u200d\u200c]", "", (s or "").strip())
        return json.dumps({
            "expected": expected, "written": written,
            "matches": norm(written) == norm(expected),
            "note": "Nukta and zero-width marks are ignored; they are not errors.",
        }, ensure_ascii=False)

    return [get_module, get_reading, get_handwriting, get_typed_answer,
            get_gaps, check_hindi]

### §4.3 — the instructions

This is the text to edit when you want the agent to behave differently. The rules
below the fold are the ones the spec names as non-negotiable.

In [ ]:
INSTRUCTIONS = """
You grade homework for Wazir Education Society. The students are children learning
English. You are given what offline pre-processing extracted — a transcript, and the
text read off a handwritten page. You never see the video or the photograph, and you
must not ask for them.

ORDER OF WORK
1. Call `get_module` first, every time. Its grading_instructions are the rubric for
   THIS task. Do not assume a rubric.
2. Call `get_gaps`. Anything listed there is something WE could not do. It must never
   lower a score.
3. Call `get_reading`, `get_handwriting` and `get_typed_answer` to see what the
   student actually produced. Any of the three may be empty.
4. Score only: speed, vocabulary, completion, accuracy.

Scale: 1 Needs Support - 2 Emerging - 3 Developing - 4 Proficient - 5 Exemplary.

GRADE WHAT IS THERE. If only the page arrived, grade the page and say the video is
missing. A missing piece is reported, never used to drag down the pieces that came.

COMPLETENESS IS REPORTED, NOT SCORED. Put the figure in `completeness_note` and in the
feedback — "12 of the 20 rows were filled". Do not quietly lower a parameter because
less was covered; a deduction hidden inside a number is one nobody can argue with.

EVIDENCE IS REQUIRED. Every score must name what you observed and which part of the
module's instructions it is measured against. "Read at 132 words per minute, which a
listener could follow" is evidence. "Good pace" is not.

ABSTAIN RATHER THAN GUESS. No transcript means speed and vocabulary cannot be judged:
set them to null and say why. An abstention is a good answer. An invented number is
not. Never return a zero.

AN ACCENT IS NEVER AN ERROR. Indian and regional accents and dialects are not marked
down, ever. Judge whether the student was understandable. If a dialect form looks like
a mistake, say so in for_the_teacher and let a teacher decide.

JUDGE THE WORK, NEVER THE CHILD. You may not conclude anything about a student's
intelligence, ability, effort, motivation, honesty, emotional state, home
circumstances, caste, religion, gender or any disability. Not in the feedback, not in
the evidence, not in the teacher note.

FEEDBACK. Two to four short sentences, English then Hindi. Warm, never harsh. One
specific thing they did well FIRST, then one concrete thing to do next. Plain words.
Never compare them to another student.
""".strip()

print(INSTRUCTIONS[:300], "...")

### §4.4 — the runner

In [ ]:
async def run_grading_agent(module: Module, pre: PreProcessed, sub: Submission,
                            model: str = "gpt-4o") -> GradingResult:
    from agents import Agent, Runner

    agent = Agent(
        name="WES homework grader",
        model=model,
        instructions=INSTRUCTIONS,
        tools=build_tools(module, pre),
        output_type=GradingResult,
    )

    result = await Runner.run(
        agent,
        f"Grade submission {sub.submission_id} from student {sub.student_id} "
        f"for module '{sub.module_id}'. Start by reading the module.",
    )
    return result.final_output          # typed as GradingResult, not free text

In [ ]:
# §0 already called load_dotenv(). This just confirms the key arrived.
assert os.environ.get("OPENAI_API_KEY"), (
    "No OPENAI_API_KEY. Create a .env file next to this notebook containing
"
    "    OPENAI_API_KEY=sk-...
"
    "then re-run §0.")
print("Key loaded from .env — ready to run the agent.")

## §5 — BOX 5 · Score + Evidence + Feedback

Before any of it reaches a teacher, it goes through checks that are ordinary code
rather than instructions to the model. This is the difference between a guarantee
and a hope about a prompt.

Three of these **block** the whole result rather than editing it. A system that
quietly rewrites a harsh sentence gives you no way to notice it happened.

In [ ]:
class Blocked(Exception):
    """The result must not reach a teacher or a student."""


# Phrases that judge the child rather than the work. Both the contraction and the
# expansion of each, because a model writing formal English says "did not bother"
# where a person would type "didn't".
FORBIDDEN = [
    "lazy", "careless", "not trying", "didn't bother", "did not bother",
    "unmotivated", "disinterested", "uninterested", "does not care", "doesn't care",
    "no effort", "made no effort", "did not try", "didn't try",
    "slow learner", "weak student", "bright child", "not very bright", "low iq",
    "cheating", "copied from", "dishonest", "lying",
    "poor family", "uneducated parents", "neglected", "underprivileged",
    "seems bored", "looks bored", "not paying attention",
]


def validate(result: GradingResult) -> GradingResult:
    """Everything that must hold whatever the model returned."""
    problems: list[str] = []

    # 1. No judgement of the child, anywhere it could be read.
    for label, text in (("feedback_en", result.feedback_en),
                        ("feedback_hi", result.feedback_hi),
                        ("for_the_teacher", result.for_the_teacher),
                        ("evidence", " ".join(s.evidence for s in result.scores))):
        low = (text or "").lower()
        hits = [p for p in FORBIDDEN if p in low]
        if hits:
            problems.append(f"{label} judges the child: {hits}")

    kept: list[Score] = []
    for s in result.scores:
        # 2. A parameter we do not grade is dropped, not published.
        if s.parameter not in PARAMETERS:
            continue
        # 3. Out of range becomes an abstention, never a clamp. If the model
        #    returned 7 we do not know what it meant, and inventing a 5 is worse.
        if s.score is not None and not (1 <= s.score <= 5):
            s = s.model_copy(update={
                "score": None,
                "abstained_because": f"Returned {s.score}, which is outside 1-5."})
        # 4. Nothing may be a zero.
        if s.score == 0:
            problems.append(f"{s.parameter} came back as 0.")
        # 5. A score with no evidence is rejected.
        if s.score is not None and not s.evidence.strip():
            problems.append(f"{s.parameter} has a score but no evidence.")
        kept.append(s)

    if problems:
        raise Blocked("; ".join(problems))

    return result.model_copy(update={"scores": kept})


@dataclass
class Graded:
    """The finished thing: what a teacher sees, and what gets stored."""
    submission_id: str
    student_id: str
    module_id: str
    scores: list[dict]
    completeness_note: str
    feedback_en: str
    feedback_hi: str
    for_the_teacher: str
    overall: Optional[float]
    graded_at: str
    gaps: list[str]
    authored_by: str = "agent"


def finalise(result: GradingResult, sub: Submission, pre: PreProcessed) -> Graded:
    result = validate(result)
    judged = [s.score for s in result.scores if s.score is not None]
    return Graded(
        submission_id=sub.submission_id,
        student_id=sub.student_id,
        module_id=sub.module_id,
        scores=[s.model_dump() for s in result.scores],
        completeness_note=result.completeness_note,
        feedback_en=result.feedback_en,
        feedback_hi=result.feedback_hi,
        for_the_teacher=result.for_the_teacher,
        # Abstentions leave the denominator. They never quietly become a low mark.
        overall=round(sum(judged) / len(judged), 2) if judged else None,
        graded_at=datetime.now(timezone.utc).isoformat(timespec="seconds"),
        gaps=list(pre.gaps),
    )

### §5 — watching the checks fire

Worth running even without an API key: this is what the safeguards actually do.

In [ ]:
ok = GradingResult(
    scores=[Score(parameter="speed", score=4, evidence="135 wpm, a listener could follow."),
            Score(parameter="completion", score=3, evidence="12 of 20 rows filled.")],
    completeness_note="12 of the 20 rows were filled in.",
    feedback_en="You read clearly and used all three new words. Try to finish the last rows next time.",
    feedback_hi="आपने साफ़ पढ़ा और तीनों नए शब्द इस्तेमाल किए। अगली बार बाकी पंक्तियाँ भी पूरी करें।")

print("1. a clean result passes:",
      [s.parameter for s in validate(ok).scores])

sneaky = ok.model_copy(update={"scores": ok.scores + [
    Score(parameter="confidence", score=2, evidence="Sounded hesitant.")]})
print("2. out-of-scope parameter dropped ->",
      [s.parameter for s in validate(sneaky).scores])

wild = ok.model_copy(update={"scores": [
    Score(parameter="speed", score=7, evidence="Very fast.")]})
print("3. a 7 becomes an abstention ->", validate(wild).scores[0].score,
      "|", validate(wild).scores[0].abstained_because)

harsh = ok.model_copy(update={"feedback_en": "You clearly did not bother with this."})
try:
    validate(harsh)
    print("4. PUBLISHED — that would be a bug")
except Blocked as e:
    print("4. blocked ->", e)

## §6 — BOX 6 · Stored in DB

One line of JSON per graded submission. Score, evidence, feedback, date, module
and student — the fields the report card and the trend view will need later.

A JSONL file is the right store while we are building against a local copy of the
data. The Supabase version drops in behind the same two functions.

In [4]:
STORE = DATA / "graded.jsonl"


def save(g: Graded) -> None:
    with open(STORE, "a", encoding="utf-8") as fh:
        fh.write(json.dumps(asdict(g), ensure_ascii=False) + "\n")


def load_all(student_id: Optional[str] = None) -> list[dict]:
    if not STORE.exists():
        return []
    rows = [json.loads(line) for line in STORE.read_text(encoding="utf-8").splitlines() if line.strip()]
    return [r for r in rows if student_id is None or r["student_id"] == student_id]


# TODO(Supabase): same two functions, different body.
#   def save(g): supabase.table("gradings").insert(asdict(g)).execute()
#   def load_all(sid): supabase.table("gradings").select("*").eq("student_id", sid).execute().data
#
# Consent gate goes in front of BOX 3, not here — a consent check after the media
# has already been read is not a consent check.

print("Store:", STORE, "|", len(load_all()), "rows so far")

Store: C:\Users\dpatankar\Grading-Agent_Education-Society\MVP\pipeline_data\graded.jsonl | 0 rows so far


## §7 — BOX 7 · Show on Dashboard

The teacher's view. Two rules it has to respect: every score shows the reason
beside it, and an abstention shows as *not assessed* rather than as a blank or a
dash that reads like a zero.

In [ ]:
from IPython.display import HTML, display

def dashboard(rows: list[dict]) -> None:
    if not rows:
        display(HTML("<p><em>Nothing graded yet.</em></p>"))
        return

    html = ["<style>",
            ".dash{font-family:system-ui,sans-serif;font-size:14px;max-width:900px}",
            ".dash table{border-collapse:collapse;width:100%;margin-bottom:8px}",
            ".dash th{text-align:left;font-size:11px;letter-spacing:.08em;",
            "text-transform:uppercase;color:#667;border-bottom:1px solid #ccd;padding:4px 10px 4px 0}",
            ".dash td{padding:7px 10px 7px 0;border-bottom:1px solid #eef;vertical-align:top}",
            ".dash .na{color:#889;font-style:italic}",
            ".dash .why{color:#556;font-size:13px}",
            ".dash .card{border:1px solid #dde;border-radius:6px;padding:14px 16px;margin-bottom:14px}",
            ".dash .fb{background:#f6f8f7;padding:10px 12px;border-radius:4px;margin-top:8px}",
            ".dash .gap{color:#8a5a12;font-size:12.5px}",
            "</style><div class='dash'>"]

    for r in rows:
        overall = r["overall"] if r["overall"] is not None else "—"
        html.append(f"<div class='card'><b>{r['student_id']}</b> · {r['module_id']} "
                    f"· <b>overall {overall}</b> "
                    f"<span class='na'>({r['authored_by']}, {r['graded_at'][:10]})</span>")
        html.append("<table><tr><th>Parameter</th><th>Score</th><th>Why</th></tr>")
        for s in r["scores"]:
            if s["score"] is None:
                val = "<span class='na'>not assessed</span>"
                why = s.get("abstained_because") or s["evidence"]
            else:
                val = f"<b>{s['score']}</b>/5"
                why = s["evidence"]
            html.append(f"<tr><td>{s['parameter']}</td><td>{val}</td>"
                        f"<td class='why'>{why}</td></tr>")
        html.append("</table>")
        html.append(f"<div class='fb'>{r['feedback_en']}<br>{r['feedback_hi']}</div>")
        if r.get("completeness_note"):
            html.append(f"<div class='why' style='margin-top:6px'>{r['completeness_note']}</div>")
        if r.get("for_the_teacher"):
            html.append(f"<div class='why' style='margin-top:6px'><b>For the teacher:</b> "
                        f"{r['for_the_teacher']}</div>")
        for g in r.get("gaps", []):
            html.append(f"<div class='gap'>⚠ {g}</div>")
        html.append("</div>")

    html.append("</div>")
    display(HTML("".join(html)))


# A worked example so you can see the layout before running a real grading.
dashboard([asdict(finalise(
    GradingResult(
        scores=[
            Score(parameter="speed", score=4,
                  evidence="135 words per minute — a listener could follow comfortably."),
            Score(parameter="vocabulary", score=5,
                  evidence="Used all three target words from the module list."),
            Score(parameter="completion", score=3, evidence="12 of 20 rows filled."),
            Score(parameter="accuracy", score=None,
                  evidence="Only one row could be read with confidence.",
                  abstained_because="Too few readable rows to mark fairly."),
        ],
        completeness_note="12 of the 20 rows were filled in (60%).",
        feedback_en="You read clearly and said every one of the new words. "
                    "You filled 12 of the 20 rows — let's finish the rest next time.",
        feedback_hi="आपने साफ़ पढ़ा और सभी नए शब्द बोले। "
                    "आपने 20 में से 12 पंक्तियाँ भरीं — अगली बार बाकी भी पूरी करें।",
        for_the_teacher="One Hindi row was too unclear to read."),
    SUBMISSION, PRE))])

## §8 — End to end

Every box, in order, on one submission. Needs an API key and `openai-agents`.

Note there is no `asyncio.run` — Jupyter is already running an event loop, so
`await` at the top level is correct here and would be a syntax error in a plain
`.py` script.

In [ ]:
async def run_pipeline(sub: Submission, module: Module) -> Optional[Graded]:
    pre = preprocess(sub, module)                      # BOX 3
    print("pre-processing done.", len(pre.gaps), "gap(s)")

    result = await run_grading_agent(module, pre, sub)  # BOX 4
    print("agent returned", len(result.scores), "score(s)")

    try:
        graded = finalise(result, sub, pre)             # BOX 5
    except Blocked as exc:
        print("BLOCKED — nothing published:", exc)
        return None

    save(graded)                                        # BOX 6
    dashboard([asdict(graded)])                         # BOX 7
    return graded


graded = await run_pipeline(SUBMISSION, MODULE)

---

### Where to edit what

| To change… | Edit |
|---|---|
| the homework being graded | §1 `MODULE` |
| which submission runs | §2 `SUBMISSION` |
| how handwriting is read | §3b `read_handwriting` |
| how the reading is transcribed | §3c `speech_to_text` |
| what the agent can find out | §4.2 `build_tools` |
| how the agent is told to grade | §4.3 `INSTRUCTIONS` |
| which parameters are in scope | §4.1 `PARAMETERS` |
| what is never allowed through | §5 `FORBIDDEN`, `validate` |
| where results are stored | §6 `save` / `load_all` |
| what the teacher sees | §7 `dashboard` |

### Still to do

1. **§3b OCR is not wired up.** Until it is, handwriting parameters abstain.
2. **Consent.** A consent check belongs immediately in front of BOX 3 — after the
   media has been read is not a consent check. Nothing here creates a record yet.
3. **Teacher edits.** The dashboard shows results; it cannot yet edit them. The
   spec requires a teacher override before anything is final.
4. **The module comes from a literal in §1**, not the portal.